<a href="https://colab.research.google.com/github/zhabibi-z/house_price_prediction/blob/main/HousePricePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Operating Context and Setup

Based on the prompt, here are the environment settings:

- **Hardware:** 8 vCPU / 32 GB RAM / no GPU
- **AutoGluon version:** 1.x (will use the latest available in Colab, but conceptually pin to 1.x)
- **Total wall-clock budget:** 1 hour
- **Reproducibility:** `SEED = 42`

In [ ]:
# Library version pins (conceptual, as Colab often has recent versions)
# If specific versions are required, uncomment and run pip install
!pip install -q autogluon.tabular[extra]==1.2 # Pin to major 1.x release

import pandas as pd
import numpy as np
import autogluon.tabular as agt

# Reproducibility seed
SEED = 42
np.random.seed(SEED)

# Column names
ID_COL = 'Id'
TARGET_COL = 'SalePrice'
LOG_TARGET_COL = 'log_SalePrice'


### Load Data

Loading `train.csv` and `test.csv` into pandas DataFrames, setting the `Id` column as the index. This aligns with Phase 1 of the prompt: "Load with `Id` as the index."

### Download Data from Kaggle

The dataset `house-prices-advanced-regression-techniques` needs to be downloaded from Kaggle. You'll need to set up your Kaggle API credentials to do this. Follow the instructions [here](https://www.kaggle.com/docs/api#authentication) to get your `kaggle.json` file. Then, upload it to your Colab environment by clicking the folder icon on the left panel, then the upload icon, and selecting `kaggle.json`.

In [ ]:
# Install Kaggle API client
!pip install -q kaggle

# Make a directory for Kaggle and move the API key there
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API setup complete.")

In [ ]:
# Download the dataset from Kaggle
!kaggle competitions download -c house-prices-advanced-regression-techniques

print("Dataset downloaded.")

In [ ]:
# Unzip the downloaded data file
!unzip -o house-prices-advanced-regression-techniques.zip -d .

print('Data files unzipped successfully.')

In [ ]:
# Load the training and test data from local files
!ls -F
train_df = pd.read_csv('train.csv', index_col=ID_COL)
test_df = pd.read_csv('test.csv', index_col=ID_COL)

print('Data loaded successfully from local files.')
print(f"Train data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

display(train_df.head())
display(test_df.head())

## PHASE 1 — Sanitization & dirty-value correction

### 1.1 Target variable extraction and transformation

Extract `SalePrice`, apply `np.log1p`, and drop it from the training frame. This creates the `LOG_TARGET_COL` for model training.

In [ ]:
if not train_df.empty:
    y_train = np.log1p(train_df[TARGET_COL]) # Apply log1p transformation to the target
    train_df = train_df.drop(columns=[TARGET_COL]) # Drop original SalePrice from training features
    print(f"Target variable '{TARGET_COL}' extracted and transformed to '{LOG_TARGET_COL}'.")
    print(f"Original train_df shape: {train_df.shape}")
    print(f"Target (y_train) shape: {y_train.shape}")
else:
    print("Train DataFrame is empty, cannot process target variable.")

### 1.2 Structural NA to 'None' & Numeric Missingness & Dirty Values & Outlier Removal

I will define a function `preprocess_phase1` to encapsulate the remaining steps of Phase 1, ensuring it can be applied consistently to both training and test data, while respecting the train-only rules for certain operations.

This function will address:
1.  **Structural NA → "None":** for specified categorical features.
2.  **Numeric missingness:** handle `LotFrontage`, `GarageYrBlt`, `MasVnrArea`, and basement/garage numeric fields.
3.  **Dataset-specific dirty values:** correct `GarageYrBlt == 2207`, clamp negative age features, and drop `Utilities`.
4.  **Outlier removal:** `GrLivArea > 4000` and `SalePrice < 300000` (applied to train only outside this function).

In [ ]:
def preprocess_phase1(df, is_train=False, train_median_lotfrontage=None):
    # Create a copy to avoid modifying the original DataFrame
    df = df.copy()

    # 1. Structural NA -> 'None' for categorical features
    # These NaNs indicate absence, so map to 'None' string
    structural_na_cols = [
        'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType',
        'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond',
        'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'
    ]
    for col in structural_na_cols:
        if col in df.columns:
            df[col] = df[col].fillna('None')

    # 2. Numeric missingness
    # LotFrontage: impute by neighborhood median (calculated on train only)
    if 'LotFrontage' in df.columns:
        if is_train:
            # Calculate median on train data for each neighborhood
            train_median_lotfrontage = df.groupby('Neighborhood')['LotFrontage'].transform('median')
            df['LotFrontage'] = df['LotFrontage'].fillna(train_median_lotfrontage)
            # If any remain (e.g., Neighborhood not in train), fill with overall median
            df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
        else:
            # Use the train_median_lotfrontage passed from the training set
            if train_median_lotfrontage is not None:
                # For test, fill with the median of the *overall train set* if neighborhood median is not applicable
                df['LotFrontage'] = df['LotFrontage'].fillna(train_median_lotfrontage)
            else:
                # Fallback if no train median is provided (shouldn't happen in proper flow)
                df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median()) # Using test median as last resort

    # GarageYrBlt, MasVnrArea, basement/garage numeric fields
    # Fill with 0 for these where 0 often means 'no garage', 'no masonry veneer', 'no basement'
    numeric_na_fill_zero = [
        'GarageYrBlt', 'GarageArea', 'GarageCars', 'MasVnrArea',
        'BsmtFinSF1', 'BsmtFinSF2', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtUnfSF', 'TotalBsmtSF'
    ]
    for col in numeric_na_fill_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # MasVnrType: If MasVnrArea is 0, type should be 'None'
    if 'MasVnrType' in df.columns:
        df.loc[df['MasVnrArea'] == 0, 'MasVnrType'] = df.loc[df['MasVnrArea'] == 0, 'MasVnrType'].fillna('None')
        df['MasVnrType'] = df['MasVnrType'].fillna('None') # Fill any remaining NaNs

    # Electrical: fill with mode, as it's a critical feature and has only 1 missing value in train
    if 'Electrical' in df.columns:
        df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

    # 3. Dataset-specific dirty values
    # Correct GarageYrBlt == 2207 typo
    if 'GarageYrBlt' in df.columns:
        df.loc[df['GarageYrBlt'] == 2207, 'GarageYrBlt'] = df['YearBuilt'] # Assume typo means it should be YearBuilt

    # Utilities: flag as near-constant and drop it
    if 'Utilities' in df.columns:
        df = df.drop(columns=['Utilities'])

    return df, train_median_lotfrontage

# Apply preprocessing to training data first to get imputation values
if not train_df.empty:
    train_df_processed, train_median_lotfrontage_val = preprocess_phase1(train_df, is_train=True)
    test_df_processed, _ = preprocess_phase1(test_df, is_train=False, train_median_lotfrontage=train_median_lotfrontage_val)
    print("Phase 1 preprocessing applied to train and test data.")
    print(f"Processed train_df shape: {train_df_processed.shape}")
    print(f"Processed test_df shape: {test_df_processed.shape}")
    display(train_df_processed.head())
    display(test_df_processed.head())
else:
    print("Train DataFrame is empty, skipping Phase 1 preprocessing.")

### 1.3 Outlier removal (attributed correctly)

Applying the community heuristic: drop rows where `GrLivArea > 4000` **and** `SalePrice < 300000`. This is applied to the **training data only** and uses the original `SalePrice` for the condition, so I need `y_train` to filter. After filtering, I'll update `train_df_processed` and `y_train`.

In [ ]:
if not train_df.empty and not y_train.empty:
    # Temporarily combine to filter, then separate again
    temp_train_df = train_df_processed.copy()
    temp_train_df['original_SalePrice'] = np.expm1(y_train) # Invert log1p for the condition

    # Identify outliers
    outlier_condition = (temp_train_df['GrLivArea'] > 4000) & (temp_train_df['original_SalePrice'] < 300000)
    outliers_to_drop = temp_train_df[outlier_condition]

    if not outliers_to_drop.empty:
        print(f"Found {len(outliers_to_drop)} outliers to remove from training data based on GrLivArea and SalePrice condition.")
        print("Outliers identified (GrLivArea > 4000 AND original_SalePrice < 300000):")
        display(outliers_to_drop[['GrLivArea', 'original_SalePrice']])

        # Drop outliers from both features and target
        train_df_processed = train_df_processed.drop(outliers_to_drop.index)
        y_train = y_train.drop(outliers_to_drop.index)
        print(f"Training data shape after outlier removal: {train_df_processed.shape}")
        print(f"Target (y_train) shape after outlier removal: {y_train.shape}")
    else:
        print("No outliers found to remove based on the specified GrLivArea and SalePrice condition.")

    del temp_train_df # Clean up temporary DataFrame
else:
    print("Train DataFrame or target is empty, skipping outlier removal.")

## PHASE 2 — Feature Engineering

Now that the initial sanitization and dirty-value correction are complete, we will proceed with feature engineering. This phase focuses on creating new features or transforming existing ones to improve the model's predictive power. We'll identify features that can be combined, split, or modified to better represent the underlying patterns in the data.


### 2.1 Create Age Features

New features are created by calculating the age of the house and garage at the time of sale, and the age since remodeling.


In [ ]:
import scipy.stats as stats
import pandas as pd
import numpy as np # Ensure numpy is imported for np.log1p and np.expm1

# --- BEGIN: Phase 1 Preprocessing (Copied from a3dbf17f and 54379ebc for re-initialization) ---

# Define constants locally or ensure they are globally available and correct
ID_COL = 'Id'
TARGET_COL = 'SalePrice'
LOG_TARGET_COL = 'log_SalePrice'

# Re-load the original dataframes to ensure 'SalePrice' is present in train_df
# and that test_df is also in its initial state.
# This makes this preprocessing block robust to previous modifications of global train_df/test_df.
current_train_df_raw = pd.read_csv('train.csv', index_col=ID_COL)
current_test_df_raw = pd.read_csv('test.csv', index_col=ID_COL)


def preprocess_phase1(df, is_train=False, train_median_lotfrontage=None):
    df = df.copy()
    structural_na_cols = [
        'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType',
        'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond',
        'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'
    ]
    for col in structural_na_cols:
        if col in df.columns:
            df[col] = df[col].fillna('None')

    if 'LotFrontage' in df.columns:
        if is_train:
            train_median_lotfrontage = df.groupby('Neighborhood')['LotFrontage'].transform('median')
            df['LotFrontage'] = df['LotFrontage'].fillna(train_median_lotfrontage)
            df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
        else:
            if train_median_lotfrontage is not None:
                # Need to align the series by index if train_median_lotfrontage is a series
                df['LotFrontage'] = df['LotFrontage'].fillna(train_median_lotfrontage)
                # Fallback to overall median if still NaN (e.g., Neighborhood not seen in train)
                df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
            else:
                df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

    numeric_na_fill_zero = [
        'GarageYrBlt', 'GarageArea', 'GarageCars', 'MasVnrArea',
        'BsmtFinSF1', 'BsmtFinSF2', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtUnfSF', 'TotalBsmtSF'
    ]
    for col in numeric_na_fill_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    if 'MasVnrType' in df.columns:
        df.loc[df['MasVnrArea'] == 0, 'MasVnrType'] = df.loc[df['MasVnrArea'] == 0, 'MasVnrType'].fillna('None')
        df['MasVnrType'] = df['MasVnrType'].fillna('None')

    if 'Electrical' in df.columns:
        df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

    if 'GarageYrBlt' in df.columns:
        df.loc[df['GarageYrBlt'] == 2207, 'GarageYrBlt'] = df['YearBuilt']

    if 'Utilities' in df.columns:
        df = df.drop(columns=['Utilities'])

    return df, train_median_lotfrontage

# Now use the re-loaded raw dataframes
if not current_train_df_raw.empty:
    y_train = np.log1p(current_train_df_raw[TARGET_COL]) # This y_train will have 1460 rows
    initial_train_df_for_processing = current_train_df_raw.drop(columns=[TARGET_COL])
else:
    print("Original train_df is not available or empty, cannot proceed with Phase 1 preprocessing.")
    # This scenario should not happen if current_train_df_raw was successfully loaded.

# Apply preprocessing to training data first to get imputation values
if 'initial_train_df_for_processing' in locals() and not initial_train_df_for_processing.empty:
    train_df_processed, train_median_lotfrontage_val = preprocess_phase1(initial_train_df_for_processing, is_train=True)
    test_df_processed, _ = preprocess_phase1(current_test_df_raw.copy(), is_train=False, train_median_lotfrontage=train_median_lotfrontage_val)
    print("Phase 1 preprocessing applied to train and test data.")
    print(f"Processed train_df shape: {train_df_processed.shape}")
    print(f"Processed test_df shape: {test_df_processed.shape}")
else:
    print("Initial Train DataFrame for processing is empty, skipping Phase 1 preprocessing.")

# Outlier removal (from cell 54379ebc)
if 'train_df_processed' in locals() and 'y_train' in locals() and not train_df_processed.empty and not y_train.empty:
    temp_train_df = train_df_processed.copy()
    temp_train_df['original_SalePrice'] = np.expm1(y_train)

    outlier_condition = (temp_train_df['GrLivArea'] > 4000) & (temp_train_df['original_SalePrice'] < 300000)
    outliers_to_drop = temp_train_df[outlier_condition]

    if not outliers_to_drop.empty:
        print(f"Found {len(outliers_to_drop)} outliers to remove from training data based on GrLivArea and SalePrice condition.")
        train_df_processed = train_df_processed.drop(outliers_to_drop.index)
        y_train = y_train.drop(outliers_to_drop.index)
        print(f"Training data shape after outlier removal: {train_df_processed.shape}")
        print(f"Target (y_train) shape after outlier removal: {y_train.shape}")
    else:
        print("No outliers found to remove based on the specified GrLivArea and SalePrice condition.")

    del temp_train_df
elif 'train_df_processed' in locals() and 'y_train' in locals() and (train_df_processed.empty or y_train.empty):
    print("Train DataFrame or target is empty after Phase 1 preprocessing, skipping outlier removal.")
else:
    print("train_df_processed or y_train not defined for outlier removal. This should not happen if Phase 1 was correctly run.")

# --- END: Phase 1 Preprocessing ---

def create_age_features(df):
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']
    df['GarageAge'] = df['YrSold'] - df['GarageYrBlt']

    df.loc[df['GarageYrBlt'] == 0, 'GarageAge'] = 0
    df.loc[df['RemodelAge'] < 0, 'RemodelAge'] = 0
    df.loc[df['Age'] < 0, 'Age'] = 0

    df = df.drop(columns=['YearBuilt', 'YearRemodAdd', 'YrSold', 'MoSold', 'GarageYrBlt'])
    return df

train_df_processed = create_age_features(train_df_processed)
test_df_processed = create_age_features(test_df_processed)

print("Age-related features created and original year columns dropped.")
print(f"Processed train_df shape: {train_df_processed.shape}")
print(f"Processed test_df shape: {test_df_processed.shape}")

### 2.2 Create Total Area Features

Combine various square footage features to create a more comprehensive 'Total Area' feature, which is often a strong predictor of house price.


In [ ]:
def create_total_area_features(df):
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    return df

train_df_processed = create_total_area_features(train_df_processed)
test_df_processed = create_total_area_features(test_df_processed)

print("Total area and total bath features created.")
print(f"Processed train_df shape: {train_df_processed.shape}")
print(f"Processed test_df shape: {test_df_processed.shape}")

### 2.3 Polynomial Features for 'OverallQual'

Create polynomial features for 'OverallQual' to capture potential non-linear relationships with the target variable.


In [ ]:
def create_polynomial_features(df):
    if 'OverallQual' in df.columns:
        df['OverallQual_sq'] = df['OverallQual']**2
        df['OverallQual_cub'] = df['OverallQual']**3
    return df

train_df_processed = create_polynomial_features(train_df_processed)
test_df_processed = create_polynomial_features(test_df_processed)

print("Polynomial features for 'OverallQual' created.")
print(f"Processed train_df shape: {train_df_processed.shape}")
print(f"Processed test_df shape: {test_df_processed.shape}")

### 2.4 Log Transform Skewed Numerical Features

Apply `np.log1p` transformation to highly skewed numerical features to reduce their skewness, which can help models perform better by normalizing their distribution.


In [ ]:
import scipy.stats as stats

def log_transform_skewed_features(df, skew_threshold=0.75):
    numeric_feats = df.select_dtypes(include=np.number).columns
    skewed_feats = df[numeric_feats].apply(lambda x: stats.skew(x.dropna()))
    skewed_feats = skewed_feats[skewed_feats > skew_threshold].index

    for feat in skewed_feats:
        # Ensure the feature is not a constant zero or already log-transformed
        if (df[feat] > 0).any() and (df[feat].max() - df[feat].min()) > 0.001: # Check for non-constant and positive values
            df[feat] = np.log1p(df[feat])
            print(f"Log-transformed: {feat}")
    return df

train_df_processed = log_transform_skewed_features(train_df_processed)
test_df_processed = log_transform_skewed_features(test_df_processed)

print("Skewed numerical features log-transformed.")
print(f"Processed train_df shape: {train_df_processed.shape}")
print(f"Processed test_df shape: {test_df_processed.shape}")

### 2.5 Final Check of Data Types and Missing Values

Perform a final check on the processed dataframes to ensure all data types are appropriate and to identify any remaining missing values before moving to the modeling phase.


In [ ]:
print("Final check for training data:")
print(train_df_processed.info())
print("\nMissing values in training data after Feature Engineering:")
print(train_df_processed.isnull().sum()[train_df_processed.isnull().sum() > 0])

print("\nFinal check for test data:")
print(test_df_processed.info())
print("\nMissing values in test data after Feature Engineering:")
print(test_df_processed.isnull().sum()[test_df_processed.isnull().sum() > 0])

In [ ]:
# Assemble the modeling frames used by both EDA and AutoGluon:
# processed features + the log-transformed target for train; processed features for test.
train_data = train_df_processed.copy()
train_data[LOG_TARGET_COL] = y_train
test_data = test_df_processed.copy()
print(f"train_data: {train_data.shape}   test_data: {test_data.shape}")


## PHASE 3 — Exploratory Data Analysis (EDA) and Storytelling Visualizations

Although preprocessing and feature engineering have been completed, a comprehensive Exploratory Data Analysis (EDA) is crucial for understanding the data's underlying patterns, distributions, and relationships. This section aims to provide key insights through visualizations, presented with a 'storytelling' approach for stakeholders.

### 3.1 Target Variable Distribution

Understanding the distribution of our target variable (`SalePrice`) is fundamental. We'll examine its original distribution and then the distribution after `log1p` transformation, which was applied to reduce skewness and improve model performance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame for plotting that includes the original SalePrice as well
plot_data = train_data.copy()
plot_data['Original_SalePrice'] = np.expm1(plot_data[LOG_TARGET_COL])

plt.figure(figsize=(15, 6))

# Original SalePrice Distribution
plt.subplot(1, 2, 1)
sns.histplot(plot_data['Original_SalePrice'], kde=True, color='skyblue')
plt.title('Distribution of Original SalePrice')
plt.xlabel('SalePrice')
plt.ylabel('Frequency')

# Log-transformed SalePrice Distribution
plt.subplot(1, 2, 2)
sns.histplot(plot_data[LOG_TARGET_COL], kde=True, color='lightcoral')
plt.title('Distribution of Log-Transformed SalePrice')
plt.xlabel('log(1+SalePrice)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()


### Insight: Target Variable Transformation

The original `SalePrice` distribution is highly right-skewed, which is common for real estate prices. This skewness can negatively impact the performance of many regression models. By applying a `log1p` transformation, the distribution becomes much more symmetrical and closer to a normal distribution, making it more suitable for modeling.

### 3.2 Correlation Matrix of Numerical Features

Understanding the correlation between numerical features can reveal which features are strong predictors of `SalePrice` and also identify potential multicollinearity issues.

In [ ]:
plt.figure(figsize=(16, 12))
corr_matrix = train_data.select_dtypes(include=np.number).corr()
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()


### Insight: Key Feature Correlations

The heatmap provides a visual overview of relationships. We can identify features highly correlated with `log_SalePrice` (our target), such as `OverallQual`, `GrLivArea`, `GarageCars`, `GarageArea`, `TotalSF`, and `TotalBath`. High correlations between independent variables (multicollinearity) are also visible, which is common but can be handled by ensemble models like those in AutoGluon.

### 3.3 Relationship Between Key Features and Target

Let's visualize the relationship between some of the most influential features and the `log_SalePrice`.

In [ ]:
plt.figure(figsize=(18, 6))

# Total Living Area vs. Log SalePrice
plt.subplot(1, 3, 1)
sns.scatterplot(x='GrLivArea', y=LOG_TARGET_COL, data=train_data)
plt.title('GrLivArea vs. Log SalePrice')
plt.xlabel('Ground Living Area (sqft, log-transformed)')
plt.ylabel('log(1+SalePrice)')

# Overall Quality vs. Log SalePrice
plt.subplot(1, 3, 2)
sns.boxplot(x='OverallQual', y=LOG_TARGET_COL, data=train_data)
plt.title('Overall Quality vs. Log SalePrice')
plt.xlabel('Overall Material and Finish Quality')
plt.ylabel('log(1+SalePrice)')

# Total Square Footage vs. Log SalePrice
plt.subplot(1, 3, 3)
sns.scatterplot(x='TotalSF', y=LOG_TARGET_COL, data=train_data)
plt.title('TotalSF vs. Log SalePrice')
plt.xlabel('Total Square Footage (log-transformed)')
plt.ylabel('log(1+SalePrice)')

plt.tight_layout()
plt.show()


### Insight: Strong Predictive Relationships

These plots confirm our intuition: higher `GrLivArea`, `TotalSF`, and `OverallQual` generally correspond to higher `SalePrice`. The relationships appear strong and mostly linear after the log transformation, which is ideal for regression models.

### 4.4 Distribution of Key Categorical Features

Categorical features often represent important groupings. Let's look at the distribution of a few key ones and their impact on `SalePrice`.

In [ ]:
plt.figure(figsize=(20, 8))

# MSZoning vs. Log SalePrice
plt.subplot(1, 3, 1)
sns.boxplot(x='MSZoning', y=LOG_TARGET_COL, data=train_data)
plt.title('MSZoning vs. Log SalePrice')
plt.xlabel('Zoning Classification')
plt.ylabel('log(1+SalePrice)')
plt.xticks(rotation=45, ha='right')

# Neighborhood vs. Log SalePrice
plt.subplot(1, 3, 2)
sns.boxplot(x='Neighborhood', y=LOG_TARGET_COL, data=train_data)
plt.title('Neighborhood vs. Log SalePrice')
plt.xlabel('Physical Locations within Ames City Limits')
plt.ylabel('log(1+SalePrice)')
plt.xticks(rotation=90, ha='right')

# HouseStyle vs. Log SalePrice
plt.subplot(1, 3, 3)
sns.boxplot(x='HouseStyle', y=LOG_TARGET_COL, data=train_data)
plt.title('HouseStyle vs. Log SalePrice')
plt.xlabel('Style of dwelling')
plt.ylabel('log(1+SalePrice)')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()


### Insight: Categorical Impact

These box plots clearly demonstrate the significant impact of categorical features like `MSZoning`, `Neighborhood`, and `HouseStyle` on house prices. Different categories within these features correspond to distinct `log_SalePrice` distributions, with some neighborhoods and house styles commanding significantly higher values than others. This underscores the importance of these features in the model.


### Methodologies for Improvement Beyond Current Scope

While AutoGluon is a powerful tool for automated machine learning, further improvements could involve:

*   **More Advanced Feature Engineering:** Exploring interaction terms beyond simple polynomials, creating more domain-specific features (e.g., proximity to amenities, school ratings if available).
*   **Feature Selection:** Although AutoGluon handles feature importance, explicit feature selection techniques (e.g., recursive feature elimination, Lasso regularization) could be used to identify and potentially remove less important features before feeding them to AutoGluon, especially if dealing with extremely high-dimensional data.
*   **Ensemble Stacking (Manual):** While AutoGluon does internal stacking, a manual stacking ensemble with diverse base models and a meta-learner could potentially achieve marginal gains by leveraging external models or custom configurations not typically explored by AutoGluon's presets.
*   **Deep Learning Models:** For more complex, high-dimensional data, custom deep learning architectures could be explored, although they often require significant hyperparameter tuning and computational resources.

## PHASE 4 — Modeling

With our data now cleaned and features engineered, we will proceed to the modeling phase. This involves setting up AutoGluon for training, defining the problem type, and initiating the training process with the processed training and validation data.


### 4.1 AutoGluon Setup and Training

Now we'll prepare the processed data for AutoGluon's `TabularPredictor`. We'll define the problem type as 'regression' and train the predictor on our `train_df_processed` using `y_train` as the target.

In [ ]:
from autogluon.tabular import TabularPredictor

# Prepare data for AutoGluon
# AutoGluon expects the target column to be present in the DataFrame for training.
# We will add y_train back to a copy of train_df_processed for AutoGluon's input.

# Create the full training dataset for AutoGluon
train_data = train_df_processed.copy()
train_data[LOG_TARGET_COL] = y_train

# The test data used for predictions
test_data = test_df_processed.copy()

print(f"Train data for AutoGluon shape: {train_data.shape}")
print(f"Test data for AutoGluon shape: {test_data.shape}")

# Initialize TabularPredictor
predictor = TabularPredictor(
    label=LOG_TARGET_COL,
    eval_metric='root_mean_squared_error', # AutoGluon computes RMSLE by default for log-transformed targets
    path='AutogluonModels_HousePrices',
    problem_type='regression'
    # seed=SEED # for reproducibility - removed as it's not a valid argument for constructor
)

print("AutoGluon TabularPredictor initialized. Ready for training.")

### 4.2 Model Training

Train the AutoGluon model using the prepared training data. We'll specify a time limit and include advanced ensembles for potentially better performance.

In [ ]:
# Train the predictor
predictor.fit(
    train_data,
    time_limit=3600, # 1 hour budget in seconds
    presets='best_quality', # Focus on highest accuracy models
    # excluded_model_types=['XT'] # Exclude extremely randomized trees for speed if needed
)

print("AutoGluon model training initiated. This may take some time based on the time_limit.")

### 4.3 Make Predictions and Generate Submission File

Now that the model training is complete, we'll use the trained `predictor` to make predictions on the `test_data`. Since our target variable was log-transformed (`np.log1p`), we need to inverse-transform the predictions using `np.expm1` to get them back to the original `SalePrice` scale. Finally, we'll format these predictions into a `submission.csv` file suitable for Kaggle.

In [ ]:
# Get the leaderbaord to see model performance
leaderboard = predictor.leaderboard(silent=True)
display(leaderboard)

# Make predictions on the test data
predictions_log = predictor.predict(test_data)

# Inverse transform the predictions from log(1+SalePrice) to SalePrice
predictions = np.expm1(predictions_log)

# Create the submission DataFrame
submission_df = pd.DataFrame({
    ID_COL: test_data.index,
    TARGET_COL: predictions
})

# Save the submission file
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Predictions saved to {submission_filename}")
display(submission_df.head())


In [ ]:
import pandas as pd
# View the detailed leaderboard to see which models contributed to the ensemble
leaderboard = predictor.leaderboard(extra_info=True)
display(leaderboard[['model', 'score_val', 'stack_level', 'can_infer', 'fit_time']])

In [ ]:
!pip install -q catboost

# Prepare a clean version of the data without manual log-transforms on features
# We keep the log-transform on the target (SalePrice) as that is standard practice.
train_clean = current_train_df_raw.copy()
train_clean[LOG_TARGET_COL] = np.log1p(train_clean[TARGET_COL])
train_clean = train_clean.drop(columns=[TARGET_COL])

test_clean = current_test_df_raw.copy()

print("Retraining with CatBoost installed and raw features...")
predictor_clean = TabularPredictor(
    label=LOG_TARGET_COL,
    eval_metric='root_mean_squared_error',
    path='AutogluonModels_CleanRun'
).fit(
    train_clean,
    presets='best_quality',
    time_limit=1800 # 30 mins to see if we get a quick improvement
)

In [ ]:
# Generate new submission from the clean run
clean_preds_log = predictor_clean.predict(test_clean)
clean_preds = np.expm1(clean_preds_log)

submission_clean = pd.DataFrame({
    ID_COL: test_clean.index,
    TARGET_COL: clean_preds
})

submission_clean.to_csv('submission_clean.csv', index=False)
print("New 'submission_clean.csv' created. Try submitting this one!")

### Phase 5: Top 10 Optimization Strategy
We will now implement advanced feature engineering and a longer, more intensive training run.

In [ ]:
def create_top_10_features(df):
    df = df.copy()
    # Ensure required base features for interactions exist
    # Using 1stFlrSF + 2ndFlrSF + TotalBsmtSF as the basis for TotalSF
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF'].fillna(0)

    # Quality * Area Interactions
    df['TotalSF_x_Qual'] = df['TotalSF'] * df['OverallQual']
    df['GrLivArea_x_Qual'] = df['GrLivArea'] * df['OverallQual']

    # Neighborhood Wealth Indicators
    neigh_map = train_clean.groupby('Neighborhood')[LOG_TARGET_COL].median().to_dict()
    df['Neigh_Price_Index'] = df['Neighborhood'].map(neigh_map).fillna(train_clean[LOG_TARGET_COL].median())

    # Total Amenities
    df['Total_Full_Baths'] = df['FullBath'] + df['BsmtFullBath'].fillna(0)
    df['HasPool'] = df['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
    return df

train_top10 = create_top_10_features(train_clean)
test_top10 = create_top_10_features(test_clean)

print("Top 10 features created. Starting Deep Ensemble Training...")

predictor_top10 = TabularPredictor(
    label=LOG_TARGET_COL,
    eval_metric='root_mean_squared_error',
    path='AutogluonModels_Top10'
).fit(
    train_top10,
    presets='best_quality', # Focus on high quality
    time_limit=3600, # 1 hour budget
    num_stack_levels=3, # Deep stacking
    num_bag_folds=10, # Robust CV
    hyperparameters='very_light' # Focused search
)

In [ ]:
# Generate the ultimate submission
# Using the high-intensity predictor_top10 trained with deep stacking
top10_preds_log = predictor_top10.predict(test_top10)
top10_preds = np.expm1(top10_preds_log)

submission_top10 = pd.DataFrame({
    ID_COL: test_top10.index,
    TARGET_COL: top10_preds
})

submission_top10.to_csv('submission_top10.csv', index=False)
print("Ultimate 'submission_top10.csv' created and ready for download!")

# Display the top of the submission for a quick sanity check
display(submission_top10.head())

### Potential Fix: Targeted Preprocessing
If the leaderboard shows that simple models are winning, we should try a 'cleaner' run without the aggressive log-transformations on every feature, and try to fix the CatBoost/FastAI installation.